# Create Normal Spark Table with Kelp Metadata

This notebook demonstrates creating a standard Spark table (non-SDP) using `kelp.tables`.
The `gold_product_summary` model includes complex table properties (lists, dicts)
that Kelp serializes into JSON strings for Databricks table properties.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
dbutils = WorkspaceClient().dbutils()

In [ ]:
kelp_project_file = dbutils.widgets.get("kelp_project_file")
kelp_target = dbutils.widgets.get("kelp_target")

In [ ]:
from pyspark.sql import SparkSession

import kelp.tables as kt

spark = SparkSession.active()

kt.init(kelp_project_file, target=kelp_target)

In [ ]:
# Inspect the DDL and table properties generated from kelp metadata
model = kt.get_model("gold_product_summary")

print(f"Table FQN: {model.fqn}")
print(f"Table properties: {model.table_properties}")
print(f"\nDDL:\n{model.get_ddl()}")

In [ ]:
# Create the table using the generated DDL
ddl = kt.ddl("gold_product_summary")
if ddl:
    print(f"Executing DDL:\n{ddl}")
    spark.sql(ddl)
    print("Table created successfully.")

In [ ]:
# Populate the table with aggregated data from silver orders
fqn = kt.ref("gold_product_summary")
source_fqn = kt.ref("silver_orders_cleaned")

spark.sql(f"""
    INSERT OVERWRITE {fqn}
    SELECT
        product,
        SUM(quantity) AS total_quantity,
        SUM(price * quantity) AS total_revenue,
        COUNT(DISTINCT order_id) AS order_count
    FROM {source_fqn}
    GROUP BY product
""")

print(f"Data inserted into {fqn}.")